In [ ]:
import pandas as pd
from scipy.stats import chi2_contingency, fisher_exact
from statsmodels.stats.contingency_tables import Table2x2

In [ ]:
def chi_square_test(df, var1, var2):
    df = df[(df[var1] != 'Unknown') & (df[var2] != 'Unknown')]
    grouped = df.groupby([var1, var2]).size().reset_index(name='count')
    contingency_table = grouped.pivot_table(index=var2, columns=var1, values='count', aggfunc='sum', fill_value=0)
    print(f"Contingency Table (Grouped by {var1} and {var2}):")
    print(contingency_table)
    chi2, p, dof, expected = chi2_contingency(contingency_table)
    print(f"\nChi-square test p-value: {p}")
    print(f"\nChi-square test dof: {dof}")
    print(f"\nChi-square test chi2: {chi2}")

    print("\nOdds Ratios (UnderFifty vs FiftyPlus):")
    total_fiftyplus = contingency_table['FiftyPlus'].sum()
    total_underfifty = contingency_table['UnderFifty'].sum()

    or_results = []
    for level, row in contingency_table.iterrows():
        a = row['UnderFifty']
        b = row['FiftyPlus']
        c = total_underfifty - a
        d = total_fiftyplus - b

        table_2x2 = [[a, b], [c, d]]
        
        # Fisher's test for OR and p-value
        or_val, p_val = fisher_exact(table_2x2)
        
        # Confidence intervals using statsmodels
        sm_table = Table2x2(table_2x2)
        ci_low, ci_high = sm_table.oddsratio_confint()
        
        print(f"{level}: OR = {or_val:.3f} (95% CI {ci_low:.3f}–{ci_high:.3f}), p = {p_val:.4g}")
        or_results.append({
            'Level': level,
            'OddsRatio': or_val,
            'CI_lower': ci_low,
            'CI_upper': ci_high,
            'PValue': p_val})
        or_df = pd.DataFrame(or_results)
    print("")

    return contingency_table, p, chi2, dof, or_df

In [ ]:
# read in metadata 
df = pd.read_csv('/home/cporter/atlas_remake_June_20_2025/results/simple_metadata.csv')

In [ ]:
df['Overall_Stage'] = df['Overall_Stage'].replace('Not Applicable', 'Unknown')

In [ ]:
df

In [ ]:
# Run chi square and fishers for all clinical metadata 
pvalues = []

# check Sex and age cohort association 
contingency_table, p_value, chi2, dof, or_df = chi_square_test(df, 'Cohort', 'Sex')
pvalues.append({'Var1': 'Sex', 'Var2': 'Cohort', 'p-value': p_value})

# check MSI and age cohort association 
contingency_table, p_value, chi2, dof, or_df = chi_square_test(df, 'Cohort', 'MSI_v2')
pvalues.append({'Var1': 'MSI', 'Var2': 'Cohort', 'p-value': p_value})

# check Therapy and age cohort association 
contingency_table, p_value, chi2, dof, or_df = chi_square_test(df, 'Cohort', 'Therapy_v2')
pvalues.append({'Var1': 'Therapy', 'Var2': 'Cohort', 'p-value': p_value})

# check Sidedness and age cohort association 
contingency_table, p_value, chi2, dof, or_df = chi_square_test(df, 'Cohort', 'Sidedness')
pvalues.append({'Var1': 'Sidedness', 'Var2': 'Cohort', 'p-value': p_value})

# check overall stage and age cohort association 
contingency_table, p_value, chi2, dof, or_df = chi_square_test(df, 'Cohort', 'Overall_Stage')
pvalues.append({'Var1': 'Overall_Stage', 'Var2': 'Cohort', 'p-value': p_value})

pvalues_df = pd.DataFrame(pvalues)
print(pvalues_df)

from statsmodels.stats.multitest import multipletests
reject, pvals_corrected, _, _ = multipletests(pvalues_df['p-value'], method='fdr_bh')
pvalues_df['FDR Adjusted'] = pvals_corrected
print(pvalues_df)

In [ ]:
# check for MSS + LEFT only 
df = df[(df[df['MSI_v2']=='MSS: STABLE']) & (df['Sidedness']=='Left')]
df

In [ ]:
pvalues = []

# check Sex and age cohort association 
contingency_table, p_value, chi2, dof, or_df = chi_square_test(df, 'Cohort', 'Sex')
pvalues.append({'Var1': 'Sex', 'Var2': 'Cohort', 'p-value': p_value})

## check MSI and age cohort association 
#contingency_table, p_value, chi2, dof, or_df = chi_square_test(df, 'Cohort', 'MSI_v2')
#pvalues.append({'Var1': 'MSI', 'Var2': 'Cohort', 'p-value': p_value})

# check Therapy and age cohort association 
contingency_table, p_value, chi2, dof, or_df = chi_square_test(df, 'Cohort', 'Therapy_v2')
pvalues.append({'Var1': 'Therapy', 'Var2': 'Cohort', 'p-value': p_value})

# check overall stage and age cohort association 
contingency_table, p_value, chi2, dof, or_df = chi_square_test(df, 'Cohort', 'Overall_Stage')
pvalues.append({'Var1': 'Overall_Stage', 'Var2': 'Cohort', 'p-value': p_value})

pvalues_df = pd.DataFrame(pvalues)
print(pvalues_df)

from statsmodels.stats.multitest import multipletests
reject, pvals_corrected, _, _ = multipletests(pvalues_df['p-value'], method='fdr_bh')
pvalues_df['FDR Adjusted'] = pvals_corrected
print(pvalues_df)